In [ ]:
# Cell 1: Imports
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
from src import config, data_loader, arima_model

In [ ]:
# Cell 2: Load Data
df = data_loader.download_data(config.TICKER, config.START_DATE, config.END_DATE)
close_price = df['Close']

In [ ]:
# Cell 3: Stationarity Test (ADF)
def test_stationarity(timeseries):
    result = adfuller(timeseries)
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')
    if result[1] <= 0.05:
        print("Data is Stationary")
    else:
        print("Data is Non-Stationary")

test_stationarity(close_price)

In [ ]:
# Cell 4: ACF and PACF Plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(close_price, ax=ax1)
plot_pacf(close_price, ax=ax2)
plt.show()

In [ ]:
# Cell 5: Auto ARIMA Training
arima = arima_model.ArimaForecaster()
train_size = int(len(close_price) * 0.8)
train, test = close_price[:train_size], close_price[train_size:]

arima.find_optimal_params(train)
arima.train(train)

In [ ]:
# Cell 6: Forecast
preds, conf_int = arima.predict(len(test))
plt.figure(figsize=(12, 6))
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test')
plt.plot(test.index, preds, label='Prediction')
plt.fill_between(test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='k', alpha=0.1)
plt.legend()
plt.show()